# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhinavt1325/Flyrank-Internship-Capstone/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

### Task Type: Ranking / Priority Scoring
- **Decision to improve:** Content editors and SEO teams at FlyRank have thousands of published articles across client sites. They need to decide *which specific content page should be updated/refreshed FIRST* each week to recover slipping traffic and maximize restored clicks/impressions.
- **Task Type Selection:** **Ranking / Priority Scoring** (or Learning-to-Rank / Priority Score estimation).
- **Why this task type:** The business goal is not merely predicting if an article drops (binary classification), nor is it grouping articles into arbitrary buckets (clustering). The actionable decision is to produce an ordered, prioritized queue of content items per client where top items represent pages with the highest expected traffic gain from a refresh action.

In [9]:
# Code check: Total content items and client distribution in starter dataset
import pandas as pd

df_starter = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Total content items in starter dataset: {len(df_starter):,}")
print(f"Unique clients: {df_starter['client_id'].nunique()}")
print(f"Average content items per client: {len(df_starter) / df_starter['client_id'].nunique():.1f}")

Total content items in starter dataset: 30,000
Unique clients: 32
Average content items per client: 937.5


## 2. Target or proxy

### Target / Proxy Definition
- **Target:** **Observed Traffic Drop Ratio & Impact Score** measured as the relative change in clicks/impressions between recent and prior non-overlapping time windows (e.g., `clicks_last_30d - clicks_prev_30d` weighted by baseline impressions).
- **Ground Truth Origin:** **Observed Outcome** derived strictly from measured analytics/search metrics across distinct time windows (recent 30 days vs prior 30 days).
- **Target Caveat & Leakage Rule:** Hand-written product rules or flags (such as `is_declining_label`, `trend_direction`, and `trend_pct` in the starter CSV) encode pre-existing heuristic decisions. They are **never** used as input features; they serve only as baseline comparison benchmarks.

In [10]:
# Calculate observed click change (proxy target) from non-overlapping 30-day windows
df_starter['observed_click_diff'] = df_starter['clicks_last_30d'] - df_starter['clicks_prev_30d']
df_starter['observed_decline_flag'] = (df_starter['observed_click_diff'] < 0).astype(int)

print("Observed decline distribution (30d vs prev 30d):")
print(df_starter['observed_decline_flag'].value_counts(normalize=True).round(4))
print("\nTarget summary statistics (click difference):")
print(df_starter['observed_click_diff'].describe())

Observed decline distribution (30d vs prev 30d):
observed_decline_flag
0    0.7731
1    0.2269
Name: proportion, dtype: float64

Target summary statistics (click difference):
count    30000.000000
mean        -0.501233
std         11.170325
min       -678.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        528.000000
Name: observed_click_diff, dtype: float64


## 3. Success metric

### Success Metric: Precision@K and Mean Reciprocal Rank (MRR)
- **Primary Metric:** **Precision@K** (specifically **Precision@20** and **Precision@50** per client queue).
- **Definition of 'Good':** If an editor takes the top 20 or top 50 articles prioritized by our ML ranking score, **Precision@20 > 80%** means that at least 80% of recommended pages represent actual high-impact declining articles that yield positive traffic recovery upon refresh.
- **Secondary Benchmark Metric:** Comparison against the hand-written rule baseline (`is_declining_label`). The ML model must demonstrably beat the precision and rank quality of the static rule baseline on held-out client splits.

In [11]:
# Evaluate baseline rule precision@50 across clients
client_groups = df_starter.groupby('client_id')
baseline_precisions = []

for client_id, group in client_groups:
    if len(group) >= 50:
        # Rule baseline orders by trend_pct ascending
        top50_rule = group.sort_values(by='trend_pct', ascending=True).head(50)
        precision_at_50 = (top50_rule['observed_decline_flag'] == 1).mean()
        baseline_precisions.append(precision_at_50)

print(f"Mean Baseline Precision@50 across clients with >= 50 pages: {pd.Series(baseline_precisions).mean():.4f}")

Mean Baseline Precision@50 across clients with >= 50 pages: 0.1408


## 4. The unit of analysis, as a real dataframe

### Unit of Analysis
- **Unit of Analysis:** **One row = One pseudonymized content page (`content_id`)** belonging to a specific client (`client_id`).
- Below is a clean preview showing key metadata features, 90-day engagement metrics, windowed performance signals, and our target concept.

In [12]:
import pandas as pd

# Load starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Select a clean representative subset of columns representing unit of analysis
cols_to_show = [
    'content_id', 'client_id', 'content_type', 'main_intent',
    'word_count', 'content_age_days', 'impressions_last_30d',
    'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d',
    'avg_position', 'ctr'
]

# Create sketch of target column (observed 30d click decline)
df['target_observed_decline'] = (df['clicks_last_30d'] < df['clicks_prev_30d']).astype(int)

display_df = df[cols_to_show + ['target_observed_decline']].head(5)
print(f"Dataframe shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("Unit of analysis: 1 row = 1 pseudonymized content item")
display_df

Dataframe shape: 30000 rows x 45 columns
Unit of analysis: 1 row = 1 pseudonymized content item


,content_id,client_id,content_type,main_intent,word_count,content_age_days,impressions_last_30d,clicks_last_30d,impressions_prev_30d,clicks_prev_30d,avg_position,ctr,target_observed_decline
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,578,2,987,13,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,2501,2,5915,1,20.3,0.05,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,2382,1,6089,3,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,3626,22,4206,17,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,4211,10,6452,2,44.0,0.13,0


## 5. Why ML beats a fixed rule here

### Why ML Beats a Fixed Rule
1. **Multi-dimensional Signal Interaction:** A static rule (e.g., `if trend_pct < -20% and position > 10`) misses high-traffic pages slipping slightly from position 2 to 4 (losing thousands of clicks) while over-flagging low-traffic pages slipping from position 50 to 80.
2. **Non-linear & Contextual Decay:** Content decay depends simultaneously on content age, intent type (informational vs transactional decay at different rates), competition level, and engagement metrics (bounce/scroll rates). Writing manual `if-else` rules for all combinations across 32+ diverse client domains leads to fragile, unmaintainable heuristics.
3. **Continuous Priority Ranking:** Machine learning models learn non-linear feature interactions and score articles continuously based on expected recovery value rather than hard binary cutoffs.

In [13]:
# Demonstrate tangled signal interaction: Position slip impact varies dramatically by current rank tier
print("Traffic volume by position tier:")
print(df.groupby('position_tier')[['impressions_90d', 'clicks_90d']].mean().round(1))

print("\nDecline rate across content types and age tiers:")
print(pd.crosstab(df['content_type'], df['age_tier'], values=df['target_observed_decline'], aggfunc='mean').round(3))

Traffic volume by position tier:
               impressions_90d  clicks_90d
position_tier                             
deep                     931.2         0.4
page_1                  7582.1        26.6
page_3_5                4858.1         7.5
striking                3147.9        10.9
top_3                   3030.1        14.8

Decline rate across content types and age tiers:
age_tier            181-365  31-90   365+  91-180
content_type                                     
comparison article    0.038    NaN    NaN   0.063
feedly article        0.121  0.000    NaN   0.028
keyword article       0.246  0.254  0.219   0.251


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.